<div style="background-color:#e6f2ff; padding:20px; border-radius:10px;">
<img style="float:left; margin-right:20px;" src='Figures/alinco.png' width="120"/>
<h1 style="color:#000047;">Actividad 2: PipelineTokenizacionNormalizaciónStemmerLematizacion</h1>
<br style="clear:both"/>
</div>

<div style="border-left:4px solid #000047; padding:10px; margin-top:10px; background:#f5f5f5;">
<b>Objetivo:</b> Crear una clase con un pipeline que tokenice, elimine stopwords, aplique stemming y lematizacion, y mida el impacto en el vocabulario.
</div>

<div style="margin-top:10px;">
<b>Instrucciones generales:</b>
<ul>
<li>Crea tu propia clase para preprocesamiento de texto considerando Normalización de texto, Stemming/Lematización y Tokenización para un corpus en inglés y español</li>
<li>Procesa el corpus provisto de 4 textos en inglés y procesa el corpus de noticias en español</li>
<li>Calcula la reduccion de vocabulario en cada etapa</li>
<li>Visualiza los resultados con matplotlib</li>

</ul>
</div>

In [21]:
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer, MWETokenizer, RegexpTokenizer
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import stopwords
import nltk
import string 
import unicodedata
import pandas as pd
import re

#Clase para procesar texto
class NLPTextCleaningPipeline:
    def __init__(self, idioma = 'spanish', opciones_normalizar = {"urls","correos","menciones","hashtags","numeros","emojis","puntuacion","espacios"},
                 remover_acentos= True, remover_stopwords= True, 
                 normalizar_unicode=True, metodo_stem = None,
                 expresiones = None                 
                ):
       
        self.idioma = idioma
        self.stopswords = set(stopwords.words('spanish' if idioma=='spanish' else 'english'))
        
        #lenguaje_stem = idioma if idioma in SnowballStemmer.languages else 'english'
        #self.stemmer = SnowballStemmer(lenguaje_stem)
        self.lematizer = WordNetLemmatizer()
        
        self.metodo = metodo_stem
        self.opciones_a_normalizar = opciones_normalizar
        
        self.word_t = word_tokenize
        self.sent_t = sent_tokenize
        self.tweet_t = TweetTokenizer()
        self.MWE_t = MWETokenizer(expresiones or [], separator='_') #(Multi-Word Expression Tokenizer) une palabras para formar una expresión compuesta.
        self.regexp_t = RegexpTokenizer(r'\w+') #(Regular Expression Tokenizer) utiliza expresiones regulares y las separa o extrae los tokens de un texto.

        self.porter = PorterStemmer()
        self.snow_en = SnowballStemmer('english') #Stemmeing con idioma inglés
        self.snow_es = SnowballStemmer('spanish') #Stemmeing con idioma español
        self.lancaster = LancasterStemmer()
 
#=======================================================================================        
    #Función para tokenizar texto con LNTK
    def tokenizar(self, texto, metodo = "word"):
            
        if metodo == "word":      return self.word_t(texto)       
        elif metodo == "sent":    return self.sent_t(texto)
        elif metodo == "tweet":   return self.tweet_t.tokenize(texto)
        elif metodo == "regexp":  return self.regexp_t.tokenize(texto)
        elif metodo == "mwe":
            tokens = self.word_t(texto.lower())
            return self.MWE_t.tokenize(tokens)
        else:
            raise ValueError(f"Método '{metodo}' no identificado")

#=======================================================================================          
    #Función que limpia los textos - corpus 
    def Normalizar(self, texto):
        if self.opciones_a_normalizar is None:
            opciones = {"urls", "correos", "menciones", "hashtags", "numeros", "emojis", "puntuacion", "espacios"}
        if "urls"       in self.opciones_a_normalizar: texto = re.sub(r'https?://\S+', '', texto)
        if "correos"    in self.opciones_a_normalizar: texto = re.sub(r'[\w.]+@[\w.]+\.[a-zA-Z]{2,}', '', texto)
        if "menciones"  in self.opciones_a_normalizar: texto = re.sub(r'@\w+', '', texto)
        if "hashtags"   in self.opciones_a_normalizar: texto = re.sub(r'#\w+', '', texto)
        if "numeros"    in self.opciones_a_normalizar: texto = re.sub(r'\S*\d\S*', '', texto)
        if "emojis"     in self.opciones_a_normalizar: texto = re.sub(r'[^\x00-\x7Fáéíóúüñ¡¿ÁÉÍÓÚÜÑa-zA-Z \n.,!?]', '', texto)
        if "puntuacion" in self.opciones_a_normalizar: texto = re.sub(r'[^\w\sáéíóúüñÁÉÍÓÚÜÑ]', ' ', texto, flags=re.UNICODE)
        if "espacios"   in self.opciones_a_normalizar: texto = re.sub(r'\s+', ' ', texto).strip()
        
        return texto.lower()

#=======================================================================================
    #Función que Elimina tildes y diacríticos usando normalización Unicode.         
    def quitar_acentos(self, texto):      
        nfkd = unicodedata.normalize('NFKD', texto)
        return ''.join(c for c in nfkd if not unicodedata.combining(c))

#=======================================================================================   
    #Función que normaliza texto y quita acentos
    def preprocesar(self, texto):
       #1.- Normalizacion
       texto_normalizado = self.Normalizar(texto)
       #2.- Quitar acentos
       texto_normalizado=self.quitar_acentos(texto_normalizado)
       return texto_normalizado

#=======================================================================================
    #Función de Stemming con NLTK, obtener la raíz de la palabra    
    def stemmeing(self, texto, metodo_stem="snowball"):
    
        tokens = self.tokenizar(texto, "word")
        vocabulario = set(tokens)
    
        if metodo_stem == "porter":            return {self.porter.stem(t) for t in vocabulario}   
        elif metodo_stem == "snowball":    
            if self.idioma == "spanish":       return {self.snow_es.stem(t) for t in vocabulario}
            else:                              return {self.snow_en.stem(t) for t in vocabulario}    
        elif metodo_stem == "lancaster":       return {self.lancaster.stem(t) for t in vocabulario}    
        else:
            raise ValueError(f"Método '{metodo_stem}' no identificado")            
#=======================================================================================
    #Función para Lematizar con NLTK
    def lematizar(self, texto):
    
        tokens = self.tokenizar(texto, "word")
        vocabulario = set(tokens)
    
        return {self.lematizer.lemmatize(t) for t in vocabulario}           
    
#=======================================================================================    
    # def fit(self):
     #   pass
    
    #def transform(self):
   #     pass    

In [22]:
# Lematización con NLTK 
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [3]:
#  Corpus de noticias en español
noticias = [
    {"categoria": "tecnología", "texto": "Apple presentó su nuevo chip M4 con IA integrada. La presentación fue en WWDC 2026 #Apple #M4"},
    {"categoria": "tecnología", "texto": "Google lanzó Gemini Ultra, su modelo más poderoso. Supera a GPT-4 en benchmarks https://blog.google.com"},
    {"categoria": "tecnología", "texto": "OpenAI anunció GPT-5 con capacidades multimodales avanzadas @OpenAI"},
    {"categoria": "economía",   "texto": "El peso mexicano se fortalece frente al dólar: $17.50 por USD en mercados internacionales"},
    {"categoria": "economía",   "texto": "La inflación en México cayó al 3.2% en abril 2026, según INEGI #Economia"},
    {"categoria": "economía",   "texto": "Inversión extranjera directa creció 15% en el primer trimestre del año"},
    {"categoria": "ciencia",    "texto": "Científicos descubrieron nueva especie de dinosaurio en Argentina #Paleontología"},
    {"categoria": "ciencia",    "texto": "Investigadores del MIT desarrollaron batería de estado sólido con carga en 5 minutos"},
    {"categoria": "ciencia",    "texto": "La NASA confirmó presencia de agua en la luna sur mediante misión Artemis https://nasa.gov"},
    {"categoria": "salud",      "texto": "Nuevo tratamiento contra el cáncer de páncreas muestra 80% de efectividad en ensayos clínicos"},
    {"categoria": "salud",      "texto": "OMS alerta sobre aumento de casos de resistencia a antibióticos en Latinoamérica #Salud"},
    {"categoria": "salud",      "texto": "Vacuna contra el dengue disponible en farmacias de México a partir de junio 2026"},
]

In [4]:
df_noticias = pd.DataFrame(noticias)
df_noticias

,categoria,texto
0,tecnología,Apple presentó su nuevo chip M4 con IA integra...
1,tecnología,"Google lanzó Gemini Ultra, su modelo más poder..."
2,tecnología,OpenAI anunció GPT-5 con capacidades multimoda...
3,economía,El peso mexicano se fortalece frente al dólar:...
4,economía,La inflación en México cayó al 3.2% en abril 2...
5,economía,Inversión extranjera directa creció 15% en el ...
6,ciencia,Científicos descubrieron nueva especie de dino...
7,ciencia,Investigadores del MIT desarrollaron batería d...
8,ciencia,La NASA confirmó presencia de agua en la luna ...
9,salud,Nuevo tratamiento contra el cáncer de páncreas...


In [5]:
# Corpus simple en inglés
corpus_en = [
    {'Documento':'text_1','texto':'Natural language processing systems are becoming increasingly sophisticated.'},
    {'Documento':'text_2','texto':'Deep learning models have revolutionized text classification and generation tasks.'},
    {'Documento':'text_3','texto':'Tokenization is the fundamental first step in any NLP preprocessing pipeline.'},
    {'Documento':'text_4','texto':'Researchers are studying computational approaches to human language understanding.'},
]
df_corpus_en = pd.DataFrame(corpus_en)
df_corpus_en

,Documento,texto
0,text_1,Natural language processing systems are becomi...
1,text_2,Deep learning models have revolutionized text ...
2,text_3,Tokenization is the fundamental first step in ...
3,text_4,Researchers are studying computational approac...


In [6]:
nlp_cleaning_eng = NLPTextCleaningPipeline(idioma = 'english')

In [7]:
df_corpus_en['texto'][0]
nlp_cleaning_eng.Normalizar(df_corpus_en['texto'][0])
nlp_cleaning_eng.preprocesar(df_corpus_en['texto'][0])

'natural language processing systems are becoming increasingly sophisticated'

In [8]:
class_NLP = NLPTextCleaningPipeline()

In [9]:
#PROCESAR TOKENIZER, código para comprobar resultados

for noticia in noticias:

    print("Categoría:", noticia["categoria"])
    print("Texto:", noticia["texto"])

    print("\nWORD TOKENIZER:", class_NLP.tokenizar(noticia["texto"], "word"))
    print("\nTWEET TOKENIZER:", class_NLP.tokenizar(noticia["texto"], "tweet"))
    print("\nREGEXP TOKENIZER:", class_NLP.tokenizar(noticia["texto"], "regexp"))
    print("\n" + "="*100 + "\n")

Categoría: tecnología
Texto: Apple presentó su nuevo chip M4 con IA integrada. La presentación fue en WWDC 2026 #Apple #M4

WORD TOKENIZER: ['Apple', 'presentó', 'su', 'nuevo', 'chip', 'M4', 'con', 'IA', 'integrada', '.', 'La', 'presentación', 'fue', 'en', 'WWDC', '2026', '#', 'Apple', '#', 'M4']

TWEET TOKENIZER: ['Apple', 'presentó', 'su', 'nuevo', 'chip', 'M4', 'con', 'IA', 'integrada', '.', 'La', 'presentación', 'fue', 'en', 'WWDC', '2026', '#Apple', '#M4']

REGEXP TOKENIZER: ['Apple', 'presentó', 'su', 'nuevo', 'chip', 'M4', 'con', 'IA', 'integrada', 'La', 'presentación', 'fue', 'en', 'WWDC', '2026', 'Apple', 'M4']


Categoría: tecnología
Texto: Google lanzó Gemini Ultra, su modelo más poderoso. Supera a GPT-4 en benchmarks https://blog.google.com

WORD TOKENIZER: ['Google', 'lanzó', 'Gemini', 'Ultra', ',', 'su', 'modelo', 'más', 'poderoso', '.', 'Supera', 'a', 'GPT-4', 'en', 'benchmarks', 'https', ':', '//blog.google.com']

TWEET TOKENIZER: ['Google', 'lanzó', 'Gemini', 'Ultra', 

In [25]:
class_NLP = NLPTextCleaningPipeline(idioma="spanish")

In [26]:
#PROCESAR STEMMEING, código para comprobar resultados con el idioma español en un corpus español
for noticia in noticias:

    print("Categoría:", noticia["categoria"])
    print("Texto:", noticia["texto"])

    print("\nPORTER:", class_NLP.stemmeing(noticia["texto"], "porter"))
    print("\nSNOWBALL:", class_NLP.stemmeing(noticia["texto"], "snowball"))
    print("\nLANCASTER:", class_NLP.stemmeing(noticia["texto"], "lancaster"))
    print("\n" + "="*100 + "\n")

Categoría: tecnología
Texto: Apple presentó su nuevo chip M4 con IA integrada. La presentación fue en WWDC 2026 #Apple #M4

PORTER: {'#', 'presentación', 'ia', 'nuevo', 'con', 'en', 'wwdc', 'fue', 'appl', 'presentó', 'integrada', 'chip', 'la', 'su', 'm4', '.', '2026'}

SNOWBALL: {'#', 'ia', 'con', 'en', '2026', 'wwdc', 'fue', 'integr', 'present', 'chip', 'la', 'su', 'm4', '.', 'apple', 'nuev'}

LANCASTER: {'#', 'presentación', 'ia', 'nuevo', 'con', 'integrad', 'en', 'wwdc', 'fue', 'appl', 'presentó', 'chip', 'la', 'su', 'm4', '.', '2026'}


Categoría: tecnología
Texto: Google lanzó Gemini Ultra, su modelo más poderoso. Supera a GPT-4 en benchmarks https://blog.google.com

PORTER: {'a', ':', 'gemini', 'modelo', ',', 'en', 'supera', 'http', 'benchmark', 'gpt-4', '//blog.google.com', 'ultra', 'má', 'su', 'googl', '.', 'poderoso', 'lanzó'}

SNOWBALL: {'a', ':', 'gemini', 'poder', ',', 'model', 'en', 'benchmarks', 'mas', 'gpt-4', '//blog.google.com', 'lanz', 'ultra', 'su', 'googl', 'super',

In [23]:
class_NLP = NLPTextCleaningPipeline(idioma="english")

In [24]:
#PROCESAR STEMMEING, código para comprobar resultados con el idioma inglés en un corpus inglés
for corpus in corpus_en:

    print("Documento:", corpus["Documento"])
    print("Texto:", corpus["texto"])

    print("\nPORTER:", class_NLP.stemmeing(corpus["texto"], "porter"))
    print("\nSNOWBALL:", class_NLP.stemmeing(corpus["texto"], "snowball"))
    print("\nLANCASTER:", class_NLP.stemmeing(corpus["texto"], "lancaster"))
    print("\n" + "="*100 + "\n")

Documento: text_1
Texto: Natural language processing systems are becoming increasingly sophisticated.

PORTER: {'.', 'natur', 'languag', 'becom', 'are', 'system', 'process', 'sophist', 'increasingli'}

SNOWBALL: {'natur', 'languag', 'increas', 'becom', 'are', 'system', 'process', 'sophist', '.'}

LANCASTER: {'langu', 'increas', 'becom', 'system', 'nat', 'process', 'ar', '.', 'soph'}


Documento: text_2
Texto: Deep learning models have revolutionized text classification and generation tasks.

PORTER: {'gener', 'and', 'model', 'classif', 'learn', 'text', 'revolution', 'have', 'deep', '.', 'task'}

SNOWBALL: {'and', 'model', 'classif', 'learn', 'text', 'generat', 'revolution', 'have', 'deep', '.', 'task'}

LANCASTER: {'class', 'and', 'model', 'hav', 'gen', 'learn', 'revolv', 'text', 'deep', '.', 'task'}


Documento: text_3
Texto: Tokenization is the fundamental first step in any NLP preprocessing pipeline.

PORTER: {'pipelin', '.', 'fundament', 'token', 'is', 'first', 'step', 'ani', 'in',

In [27]:
class_NLP = NLPTextCleaningPipeline()

In [15]:
#PROCESAR LEMATIZAR, código para comprobar resultados
for corpus in corpus_en:
    
    print("Documento:", corpus["Documento"])
    print("Texto:", corpus["texto"])

    print("\LEMATIZAR:", class_NLP.lematizar(corpus["texto"]))
    print("\n" + "="*100 + "\n")

Documento: text_1
Texto: Natural language processing systems are becoming increasingly sophisticated.
\LEMATIZAR: {'becoming', 'increasingly', 'Natural', 'are', 'system', 'language', '.', 'sophisticated', 'processing'}


Documento: text_2
Texto: Deep learning models have revolutionized text classification and generation tasks.
\LEMATIZAR: {'and', 'revolutionized', 'generation', 'model', 'learning', 'text', 'have', 'Deep', '.', 'task', 'classification'}


Documento: text_3
Texto: Tokenization is the fundamental first step in any NLP preprocessing pipeline.
\LEMATIZAR: {'pipeline', 'fundamental', 'preprocessing', 'Tokenization', 'NLP', 'is', 'first', 'step', 'in', 'the', 'any', '.'}


Documento: text_4
Texto: Researchers are studying computational approaches to human language understanding.
\LEMATIZAR: {'human', 'studying', 'are', 'to', 'computational', 'approach', 'Researchers', 'language', '.', 'understanding'}


